In [ ]:
# papermill parameters
RUN_DIR     = "/work/runs/OE/single_texto/bm25_unigram_params__k1-0.60__b-0.35__OE"
WORK_ROOT   = "/work"
RUN_CONTEXT = "manual"


In [ ]:
# === metrics.ipynb — dual-target evaluation, gold read from the run ===
#
# What changed in S1. The gold is taken from the run record's gold_item_key and
# gold_parent_key, never from the query's own key: a BC3CAT-Syn query key is not in the
# corpus, so the old `gold == query key` assumption dropped every synthetic query silently.
# A query the evaluation cannot score is now an error, not a smaller denominator.
#
# Slices, beyond the existing has_numbers: condition, modification type, modification count,
# subchapter, and family-size tercile.

import gzip
import json
import math
import sys
from collections import defaultdict
from pathlib import Path

for _candidate in (Path.cwd(), Path(WORK_ROOT) / "src"):
    if (_candidate / "utils").is_dir() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

import numpy as np
import pandas as pd
from IPython.display import display

RUN_DIR = Path(RUN_DIR)
if not RUN_DIR.exists():
    raise FileNotFoundError(f"RUN_DIR does not exist: {RUN_DIR}")

results_files = sorted(RUN_DIR.glob("results_top*.jsonl.gz"))
if not results_files:
    raise FileNotFoundError(
        f"No results_top*.jsonl.gz in {RUN_DIR}. Run retrieve.ipynb with SAVE=True first."
    )
RESULTS_PATH = results_files[-1]

meta_path = RUN_DIR / "run_meta.json"
if not meta_path.exists():
    raise FileNotFoundError(
        f"{RUN_DIR} has no run_meta.json. Runs made before S1 do not carry one; re-run them."
    )
run_meta = json.loads(meta_path.read_text(encoding="utf-8"))

index_dir = Path(run_meta["index_dir"])
corpus_path = Path(run_meta["query_path"]).parent / f"{run_meta['collection']}_long_feats.parquet"
queries_path = Path(run_meta["query_path"])

print(f"[metrics] run        : {run_meta['run_id']}")
print(f"[metrics] collection : {run_meta['collection']} | query set: {run_meta['queryset']} | split: {run_meta['split']}")
print(f"[metrics] queries    : {run_meta['queries']} (of {run_meta['queries_before_split']} before split)")
print(f"[metrics] gold column: {run_meta['gold_column']}")

df_d = pd.read_parquet(corpus_path, columns=["item_key", "parent_key", "text_word"]).rename(
    columns={"text_word": "d_text"}
)
query_columns = [c for c in ("item_key", "text_word", "modification_types", "modification_count")]
df_q = pd.read_parquet(queries_path)
df_q = df_q[[c for c in query_columns if c in df_q.columns]].rename(columns={"text_word": "q_text"})

doc2parent = df_d.set_index(df_d["item_key"].astype(str))["parent_key"].astype(str).to_dict()
parent2docs = defaultdict(set)
for doc_key, parent_key in doc2parent.items():
    parent2docs[parent_key].add(doc_key)

q_table = df_q.set_index(df_q["item_key"].astype(str), drop=False)

# Family-size terciles, at the boundaries SPLITS.md states.
FAMILY_TERCILES = (72, 432)

def family_tercile(parent_key: str) -> str:
    size = len(parent2docs.get(parent_key, ()))
    if size <= FAMILY_TERCILES[0]:
        return "small"
    if size <= FAMILY_TERCILES[1]:
        return "medium"
    return "large"


def dcg_at_k(relevance, k):
    return sum(1.0 / math.log2(i + 1) for i, rel in enumerate(relevance[:k], start=1) if rel)


def idcg_at_10(gold_label: str, target: str) -> float:
    if target == "item":
        return dcg_at_k([1], 10)
    num_relevant = len(parent2docs.get(gold_label, ()))
    return dcg_at_k([1] * min(num_relevant, 10), 10) if num_relevant else 0.0


def iter_results(path):
    with gzip.open(path, "rt", encoding="utf-8") as gz:
        for line in gz:
            if line.strip():
                yield json.loads(line)


# ---------------- per-query records, scored against the declared gold ----------------
corpus_keys = set(df_d["item_key"].astype(str))
records = []
K_seen = None

for record in iter_results(RESULTS_PATH):
    query_key = str(record.get("query_item_key", ""))
    gold_key = str(record.get("gold_item_key", "") or query_key)

    if gold_key not in corpus_keys:
        raise KeyError(
            f"query {query_key}: gold {gold_key} is not in the corpus. A query that cannot be "
            "scored is an error, not a dropped row (INTAKE §4 breakpoint 1)."
        )

    candidates = [str(c.get("index_item_key", "")) for c in record.get("candidates", [])]
    if not candidates:
        raise ValueError(f"query {query_key}: no candidates in the run record")
    if K_seen is None:
        K_seen = len(candidates)

    gold_parent = str(record.get("gold_parent_key", "") or doc2parent.get(gold_key, ""))

    row = {"query_item_key": query_key, "gold_item_key": gold_key, "gold_parent_key": gold_parent}
    for target in ("item", "parent"):
        gold_label = gold_key if target == "item" else gold_parent
        if target == "item":
            relevance = [1 if c == gold_label else 0 for c in candidates]
        else:
            siblings = parent2docs.get(gold_label, set())
            relevance = [1 if c in siblings else 0 for c in candidates]

        rr = next((1.0 / rank for rank, rel in enumerate(relevance, start=1) if rel), 0.0)
        ideal = idcg_at_10(gold_label, target)
        row.update({
            f"{target}_acc1": int(relevance[0] == 1),
            f"{target}_rec5": int(any(relevance[:5])),
            f"{target}_rec10": int(any(relevance[:10])),
            f"{target}_rr": rr,
            f"{target}_ndcg10": (dcg_at_k(relevance, 10) / ideal) if ideal > 0 else 0.0,
        })

    # has_numbers is read off the query table, as it always was.
    q_text = q_table.loc[query_key, "q_text"] if query_key in q_table.index else ""
    row["has_numbers"] = any(ch.isdigit() for ch in str(q_text))
    row["subchapter"] = gold_parent[:3]
    row["family_tercile"] = family_tercile(gold_parent)

    if query_key in q_table.index and "modification_count" in q_table.columns:
        count = q_table.loc[query_key, "modification_count"]
        types = q_table.loc[query_key, "modification_types"]
        row["modification_count"] = int(count) if pd.notna(count) else None
        # De-duplicated, order preserved. OE_stacked_texto lists template_paraphrase twice in
        # all 4,998 records, so exploding the raw list double-counts that slice's population
        # (audit F3). modification_count counts the duplicate too, so the distinct dose is
        # one lower than the field says for every stacked record -- carried as its own slice
        # rather than corrected in place, because the field is upstream and digested (D-025).
        raw_types = list(types) if types is not None and len(types) else []
        row["modification_types"] = list(dict.fromkeys(raw_types))
        row["distinct_modification_count"] = len(row["modification_types"]) or None
        # Condition is derived, never stored (branch CLAUDE.md).
        row["condition"] = (
            f"single_{row['modification_types'][0]}"
            if row["modification_count"] == 1 and row["modification_types"]
            else "all_combined"
        )
    else:
        row["modification_count"] = None
        row["distinct_modification_count"] = None
        row["modification_types"] = []
        row["condition"] = run_meta["queryset"]

    records.append(row)

if not records:
    raise RuntimeError(f"no per-query records read from {RESULTS_PATH}")

per_query = pd.DataFrame(records)

if len(per_query) != run_meta["queries"]:
    raise ValueError(
        f"scored {len(per_query)} queries but the run says {run_meta['queries']}. "
        "No query is lost between retrieval and evaluation."
    )

print(f"[metrics] scored     : {len(per_query)} queries | K in file: {K_seen}")

# ---------------- aggregation ----------------
def aggregate(frame: pd.DataFrame, target: str) -> dict:
    return {
        "queries": float(len(frame)),
        "Acc@1": float(frame[f"{target}_acc1"].mean()),
        "Recall@5": float(frame[f"{target}_rec5"].mean()),
        "Recall@10": float(frame[f"{target}_rec10"].mean()),
        "MRR": float(frame[f"{target}_rr"].mean()),
        "nDCG@10": float(frame[f"{target}_ndcg10"].mean()),
    }


def slices(frame: pd.DataFrame):
    """(scope label, subframe) for every slice this branch reports."""
    yield "overall", frame
    if frame["has_numbers"].any():
        yield "has_numbers", frame[frame["has_numbers"]]
    if (~frame["has_numbers"]).any():
        yield "no_numbers", frame[~frame["has_numbers"]]

    for condition, group in sorted(frame.groupby("condition")):
        yield f"condition:{condition}", group

    exploded = frame.explode("modification_types")
    for mod_type, group in sorted(exploded.dropna(subset=["modification_types"]).groupby("modification_types")):
        yield f"modification_type:{mod_type}", group

    for count, group in sorted(frame.dropna(subset=["modification_count"]).groupby("modification_count")):
        yield f"modification_count:{int(count)}", group

    for count, group in sorted(
        frame.dropna(subset=["distinct_modification_count"]).groupby("distinct_modification_count")
    ):
        yield f"distinct_modification_count:{int(count)}", group

    for subchapter, group in sorted(frame.groupby("subchapter")):
        yield f"subchapter:{subchapter}", group

    for tercile, group in sorted(frame.groupby("family_tercile")):
        yield f"family_tercile:{tercile}", group


rows = []
for target in ("item", "parent"):
    for scope, group in slices(per_query):
        rows.append({"method": RUN_DIR.name, **aggregate(group, target), "scope": scope, "target": target})

report = pd.DataFrame(rows)

headline = report[report["scope"].isin(["overall", "has_numbers", "no_numbers"])]
display(headline.round(4)[["target", "scope", "queries", "Acc@1", "Recall@5", "Recall@10", "MRR", "nDCG@10"]])

print("\nSlices:")
display(
    report[~report["scope"].isin(["overall", "has_numbers", "no_numbers"])]
    .round(4)[["target", "scope", "queries", "Acc@1", "Recall@5", "Recall@10", "MRR", "nDCG@10"]]
)

# ---------------- save ----------------
for target in ("item", "parent"):
    side = report[report["target"] == target]
    side.to_json(RUN_DIR / f"metrics_single__{target}.json", orient="records", force_ascii=False, indent=2)
    side.to_csv(RUN_DIR / f"metrics_single__{target}.csv", index=False)

report.to_json(RUN_DIR / "metrics_dual.json", orient="records", force_ascii=False, indent=2)
report.to_csv(RUN_DIR / "metrics_dual.csv", index=False)
per_query.to_parquet(RUN_DIR / "results_perquery.parquet", index=False)

missed = per_query[per_query["item_acc1"] == 0]
missed[["query_item_key", "gold_item_key", "gold_parent_key", "condition"]].to_csv(
    RUN_DIR / "missed_queries.csv", index=False
)

print("\nSaved:")
for name in ("metrics_dual.json", "metrics_dual.csv", "results_perquery.parquet", "missed_queries.csv"):
    print("  ", RUN_DIR / name)
